# Tarea 1  ||  Visualización de Datos: Deadlock
* Benjamín Romero Pérez   | 202273060-9
* Benjamín Ferrada Larach | 202273061-7

## Importación de librerías


In [ ]:
import http.client
import json
import pandas as pd
import plotly.express as px
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import time
!pip install ridgeplot
import ridgeplot
import requests

### Conexión Api Deadlock

In [ ]:
# Conexión al servidor de Assets
conn = http.client.HTTPSConnection("assets.deadlock-api.com")

### Extracción de Datos para Criterios Benjamín Romero Pérez

In [ ]:
def diagnostico_heroes():
    conn.request("GET", "/v2/heroes?language=english")
    res = conn.getresponse()

    if res.status != 200:
        print(f"Error de conexión: {res.status}")
        return None

    data = json.loads(res.read().decode())
    df = pd.DataFrame(data)
    return df

df_diag_heroes = diagnostico_heroes()

def diagnostico_items_full():
    conn.request("GET", "/v2/items?language=english")
    res = conn.getresponse()

    if res.status != 200:
        print(f"Error de conexión: {res.status}")
        return None

    data = json.loads(res.read().decode())
    df = pd.DataFrame(data)

    return df

df_diag_items_full = diagnostico_items_full()



In [ ]:
# ==========================================
# EXTRACCIÓN DE DATOS - BENJAMÍN ROMERO
# ==========================================

def obtener_datos_romero():
    """
    Extrae ítems comprables con sus upgrades y propiedades,
    y héroes con su level_info para análisis de rutas de habilidades.
    """
    # --- Ítems ---
    c = http.client.HTTPSConnection("assets.deadlock-api.com")
    c.request("GET", "/v2/items?language=english")
    res = c.getresponse()
    data_items = json.loads(res.read().decode())
    c.close()

    df_items = pd.DataFrame(data_items)
    df_items = df_items[df_items['shopable'] == True].copy()

    # --- Héroes ---
    c2 = http.client.HTTPSConnection("assets.deadlock-api.com")
    c2.request("GET", "/v2/heroes?language=english")
    res2 = c2.getresponse()
    data_heroes = json.loads(res2.read().decode())
    c2.close()

    df_heroes = pd.DataFrame(data_heroes)
    df_heroes = df_heroes[
        (df_heroes['player_selectable'] == True) &
        (df_heroes['disabled'] == False) &
        (df_heroes['in_development'] == False)
    ].copy()

    return df_items, df_heroes

df_items_romero, df_heroes_romero = obtener_datos_romero()
print(f"Ítems comprables: {len(df_items_romero)}")
print(f"Héroes jugables: {len(df_heroes_romero)}")

Ítems comprables: 171
Héroes jugables: 38


In [ ]:
def criterio_romero_1(df_items, df_heroes):
    """
    Popularidad del ítem (cuántos héroes lo priorizan como 'Good')
    vs cantidad de atributos que otorga.
    """
    # --- Mismo filtro que Ferrada ---
    df = df_items[
        (df_items['cost'] > 0) &
        (df_items['cost'] != 9999) &
        (df_items['item_slot_type'].notnull()) &
        (df_items['shopable'] == True) &
        (df_items['disabled'] != True)
    ].copy()

    # --- Tier label (mismo criterio que Ferrada) ---
    def asignar_tier(costo):
        if costo <= 800:  return 'Tier 1'
        if costo <= 1600: return 'Tier 2'
        if costo <= 3200: return 'Tier 3'
        return 'Tier 4'

    df['tier_label'] = df['cost'].apply(asignar_tier)
    df['categoria'] = df['item_slot_type'].str.replace('EItemSlotType_', '', regex=False)

    # --- Número de atributos del ítem ---
    df['num_atributos'] = df['properties'].apply(
        lambda x: len(x) if isinstance(x, dict) else 0
    )

    # --- Popularidad: cuántos héroes tienen el ítem como 'Good' ---
    conteo_good = {}
    for _, heroe in df_heroes.iterrows():
        bucketing = heroe.get('item_draft_bucketing', {})
        if not isinstance(bucketing, dict):
            continue
        for class_name, info in bucketing.items():
            if isinstance(info, dict) and info.get('bucket') == 'Good':
                conteo_good[class_name] = conteo_good.get(class_name, 0) + 1

    df['popularidad_good'] = df['class_name'].map(conteo_good).fillna(0).astype(int)

    return df

df_crit1 = criterio_romero_1(df_items_romero, df_heroes_romero)

def criterio_romero_2(df_items, df_heroes):
    """
    Para cada héroe, toma sus ítems marcados como 'Good'
    y calcula el costo de:
    - Build endgame: los 12 ítems Good más caros
    - Build mínima: los 12 ítems Good más baratos
    """
    # Ítems del modo principal con costo válido
    df_items_validos = df_items[
        (df_items['cost'] > 0) &
        (df_items['cost'] != 9999) &
        (df_items['item_slot_type'].notnull()) &
        (df_items['shopable'] == True) &
        (df_items['disabled'] != True)
    ].copy()
    df_items_validos['categoria'] = df_items_validos['item_slot_type'].str.replace('EItemSlotType_', '', regex=False)

    filas = []

    for _, heroe in df_heroes.iterrows():
        bucketing = heroe.get('item_draft_bucketing', {})
        if not isinstance(bucketing, dict):
            continue

        # Obtener class_names de ítems Good de este héroe
        items_good = [cn for cn, info in bucketing.items()
                      if isinstance(info, dict) and info.get('bucket') == 'Good']

        if not items_good:
            continue

        # Filtrar solo ítems válidos del modo principal
        df_heroe = df_items_validos[df_items_validos['class_name'].isin(items_good)].copy()

        if len(df_heroe) < 12:
            continue  # Saltar héroes con menos de 12 ítems Good

        # Build endgame: 12 más caros
        build_endgame = df_heroe.nlargest(12, 'cost')
        # Build mínima: 12 más baratos
        build_minima  = df_heroe.nsmallest(12, 'cost')

        for build, tipo in [(build_endgame, 'Endgame'), (build_minima, 'Mínima')]:
            costo_total = build['cost'].sum()
            costo_weapon   = build[build['categoria'] == 'weapon']['cost'].sum()
            costo_vitality = build[build['categoria'] == 'vitality']['cost'].sum()
            costo_spirit   = build[build['categoria'] == 'spirit']['cost'].sum()

            filas.append({
                'heroe':          heroe['name'],
                'hero_type':      heroe.get('hero_type', 'unknown'),
                'tipo_build':     tipo,
                'costo_total':    costo_total,
                'costo_weapon':   costo_weapon,
                'costo_vitality': costo_vitality,
                'costo_spirit':   costo_spirit,
                'items_good':     items_good
            })

    df_builds = pd.DataFrame(filas)
    return df_builds

df_crit2 = criterio_romero_2(df_items_romero, df_heroes_romero)
print(f"Columnas encontradas: {df_crit2.columns.tolist()}")

Columnas encontradas: ['heroe', 'hero_type', 'tipo_build', 'costo_total', 'costo_weapon', 'costo_vitality', 'costo_spirit', 'items_good']


In [ ]:
def viz_unificado_dumbbell(df_crit1, df_crit2, df_heroes):
    """
    Dumbbell Chart: una línea por héroe que conecta
    costo build Mínima (punto azul) vs Endgame (punto rojo).
    Ordenado por costo Endgame descendente.
    Color del punto = popularidad promedio de ítems Good.
    """
    # --- Métricas Criterio 1 por héroe ---
    filas_c1 = []
    for _, heroe in df_heroes.iterrows():
        bucketing = heroe.get('item_draft_bucketing', {})
        if not isinstance(bucketing, dict):
            continue

        items_good = [cn for cn, info in bucketing.items()
                      if isinstance(info, dict) and info.get('bucket') == 'Good']

        df_h = df_crit1[df_crit1['class_name'].isin(items_good)]
        if df_h.empty:
            continue

        filas_c1.append({
            'heroe':           heroe['name'],
            'avg_atributos':   df_h['num_atributos'].mean(),
            'avg_popularidad': df_h['popularidad_good'].mean(),
        })

    df_c1_heroes = pd.DataFrame(filas_c1)

    # --- Métricas Criterio 2 ---
    df_endgame = df_crit2[df_crit2['tipo_build'] == 'Endgame'][['heroe', 'costo_total']].rename(columns={'costo_total': 'costo_endgame'})
    df_minima  = df_crit2[df_crit2['tipo_build'] == 'Mínima'][['heroe', 'costo_total']].rename(columns={'costo_total': 'costo_minima'})
    df_c2      = pd.merge(df_endgame, df_minima, on='heroe')
    df_c2['brecha_builds'] = df_c2['costo_endgame'] - df_c2['costo_minima']

    # --- Merge y ordenar ---
    df_unified = pd.merge(df_c1_heroes, df_c2, on='heroe')
    df_unified = df_unified.sort_values('costo_endgame', ascending=True)

    fig = go.Figure()

    # --- Líneas conectoras ---
    for _, row in df_unified.iterrows():
        fig.add_trace(go.Scatter(
            x=[row['costo_minima'], row['costo_endgame']],
            y=[row['heroe'], row['heroe']],
            mode='lines',
            line=dict(color='lightgray', width=2),
            showlegend=False,
            hoverinfo='skip'
        ))

    # --- Puntos Build Mínima ---
    fig.add_trace(go.Scatter(
        x=df_unified['costo_minima'],
        y=df_unified['heroe'],
        mode='markers',
        name='Build Mínima',
        marker=dict(
            color='#457b9d',
            size=10,
            line=dict(width=1, color='white')
        ),
        hovertemplate=(
            '<b>%{y}</b><br>'
            'Build Mínima: %{x:,.0f} almas<br>'
            '<extra></extra>'
        )
    ))

    # --- Puntos Build Endgame ---
    fig.add_trace(go.Scatter(
        x=df_unified['costo_endgame'],
        y=df_unified['heroe'],
        mode='markers',
        name='Build Endgame',
        marker=dict(
            color=df_unified['avg_popularidad'],
            colorscale='YlOrRd',
            size=14,
            line=dict(width=1, color='white'),
            colorbar=dict(
                title='Popularidad<br>Promedio<br>Ítems',
                x=1.02
            ),
            showscale=True
        ),
        hovertemplate=(
            '<b>%{y}</b><br>'
            'Build Endgame: %{x:,.0f} almas<br>'
            'Popularidad promedio ítems: %{marker.color:.1f}<br>'
            'Atributos promedio ítems: %{customdata:.1f}<br>'
            '<extra></extra>'
        ),
        customdata=df_unified['avg_atributos']
    ))

    fig.update_layout(
        title='Costo de Build por Héroe: Mínima vs Endgame<br>'
              '<sup>Color punto Endgame = Popularidad promedio de ítems Good | '
              'Línea = Brecha entre ambas builds</sup>',
        xaxis=dict(
            title='Costo Total (Almas)',
            tickformat=',.0f',
            gridcolor='rgba(200,200,200,0.3)'
        ),
        yaxis=dict(
            title='',
            tickfont=dict(size=10)
        ),
        height=900,
        legend=dict(x=0.01, y=0.99),
        plot_bgcolor='white',
        margin=dict(l=130, r=100, t=100, b=60)
    )

    fig.show()
    return df_unified

df_romero = viz_unificado_dumbbell(df_crit1, df_crit2, df_heroes_romero)

### Extracción de Datos para Criterios Benjamín Ferrada Larach

In [ ]:
# ==========================================
# 1. FUNCIÓN DE EXTRACCIÓN Y DIAGNÓSTICO
# ==========================================
def obtener_datos():
    """Extrae datos y verifica la existencia de columnas clave."""
    conn.request("GET", "/v2/items?language=english")
    res = conn.getresponse()

    if res.status != 200:
        print(f"Error de conexión: {res.status}")
        return None

    data = json.loads(res.read().decode())
    conn.close()
    df_benyen = pd.DataFrame(data)

    print("--- COLUMNAS ---")
    print(f"Columnas encontradas: {df_benyen.columns.tolist()}")

    df_benyen = df_benyen[(df_benyen['shopable'] == True)]

    return df_benyen

raw_data = obtener_datos()

--- COLUMNAS ---
Columnas encontradas: ['id', 'class_name', 'name', 'start_trained', 'image', 'image_webp', 'heroes', 'properties', 'weapon_info', 'type', 'hero', 'update_time', 'behaviours', 'description', 'upgrades', 'ability_type', 'boss_damage_scale', 'tooltip_details', 'videos', 'dependent_abilities', 'item_slot_type', 'item_tier', 'activation', 'tooltip_sections', 'is_active_item', 'shopable', 'cost', 'shop_image', 'shop_image_webp', 'disabled', 'component_items', 'imbue', 'grant_ammo_on_cast']


In [ ]:
# ==========================================
# 2. CRITERIO 1: DISTRIBUCIÓN DE COSTOS
# ==========================================
def criterio_1(df):
    """
    Criterio 1: Distribución de costos (Tier 1 a 4)
    por categoría de item (Weapon, Vitality, Spirit).
    """

    df_items = df[(df['cost'] > 0) & (df['item_slot_type'].notnull())].copy()

    def asignar_tier(costo):
        if costo <= 800: return 'Tier 1'
        if costo <= 1600: return 'Tier 2'
        if costo <= 3200: return 'Tier 3'
        return 'Tier 4'

    df_items['tier_label'] = df_items['cost'].apply(asignar_tier)
    df_items['categoria'] = df_items['item_slot_type'].str.replace('EItemSlotType_', '')

    return df_items

# ==========================================
# 3. CRITERIO 2: DENSIDAD DE ATRIBUTOS
# ==========================================
def criterio_2(df):
    """
    Criterio B: Densidad de atributos por cada alma invertida.
    """
    df['num_stats'] = df['properties'].apply(lambda x: len(x) if isinstance(x, dict) else 0)


    df['densidad_stats'] = (df['num_stats'] / df['cost']) * 1000

    return df
# ==========================================
# EJECUCIÓN DEL PROCESO
# ==========================================
if __name__ == "__main__":
    if raw_data is not None:
        df_con_tiers = criterio_1(raw_data)
        df_final = criterio_2(df_con_tiers)


        df_final.to_csv("datos_ferrada_deadlock.csv", index=False)
        print("Archivo guardado: datos_ferrada_deadlock.csv")
        df_benyen = df_final.copy()

Archivo guardado: datos_ferrada_deadlock.csv


In [ ]:
# ==========================================
# 4. FUNCIÓNES DE VISUALIZACIÓN
# ==========================================
def generar_visualizacion(df):
    print("Generando Treemap...")
    fig = px.treemap(
        df,
        path=['categoria', 'tier_label', 'name'],
        values='cost',
        color='densidad_stats',
        color_continuous_scale='Viridis',
        title="Análisis Deadlock: Economía y Eficiencia de Atributos",
        labels={'densidad_stats': 'Stats por 1k Almas'}
    )
    fig.show()

def generar_sankey_deadlock(df):
    """
    Genera un Diagrama de Sankey
    Muestra el flujo: Categoría -> Tier -> Item.
    """

    df1 = df.groupby(['categoria', 'tier_label'])['cost'].sum().reset_index()

    df2 = df[['tier_label', 'name', 'cost', 'densidad_stats']]


    nodes = list(pd.unique(df[['categoria', 'tier_label', 'name']].values.ravel('K')))
    mapping = {node: i for i, node in enumerate(nodes)}


    sources = [mapping[row['categoria']] for _, row in df1.iterrows()]
    targets = [mapping[row['tier_label']] for _, row in df1.iterrows()]
    values = df1['cost'].tolist()

    sources += [mapping[row['tier_label']] for _, row in df2.iterrows()]
    targets += [mapping[row['name']] for _, row in df2.iterrows()]
    values += df2['cost'].tolist()


    fig = go.Figure(data=[go.Sankey(
        node=dict(
            pad=15,
            thickness=20,
            line=dict(color="black", width=0.5),
            label=nodes,
            color="royalblue"
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values,

            color="rgba(173, 216, 230, 0.4)"
        )
    )])

    fig.update_layout(title_text="Flujo Económico de Deadlock: Categorías, Tiers e Items", font_size=10)
    fig.show()

def generar_coordenadas_paralelas(df):
    """
    Genera un Parallel Coordinates Plot.
    Ideal para comparar Costo vs Cantidad de Atributos vs Densidad.
    """

    df_plot = df[(df['disabled'] == False) & (df['shopable'] == True)].copy()


    df_plot['cat_id'] = df_plot['item_slot_type'].astype('category').cat.codes


    fig = px.parallel_coordinates(
        df_plot,
        color="cat_id",
        dimensions=['cost', 'num_stats', 'densidad_stats'],
        labels={
            'cost': 'Costo (Almas)',
            'num_stats': 'N° Atributos',
            'densidad_stats': 'Densidad (Stats/1k)',
            'cat_id': 'Categoría'
        },
        color_continuous_scale=px.colors.diverging.Tealrose,
        title="Comparativa Multidimensional: Costo, Atributos y Eficiencia"
    )

    fig.show()

def generar_sunburst_deadlock(df):
    """
    Genera un Sunburst Chart.
    Ideal para mostrar jerarquía de costos y eficiencia de forma circular.
    """

    df_plot = df[(df['disabled'] == False) & (df['shopable'] == True)].copy()

    df_plot['categoria'] = df_plot['item_slot_type'].str.replace('EItemSlotType_', '')


    fig = px.sunburst(
        df_plot,
        path=['categoria', 'tier_label', 'name'],
        values='cost',
        color='densidad_stats',
        color_continuous_scale='Viridis',
        title="Explosión Solar: Jerarquía Económica y Eficiencia en Deadlock",
        labels={
            'categoria': 'Categoría',
            'tier_label': 'Nivel',
            'cost': 'Costo (Almas)',
            'densidad_stats': 'Stats/1k Almas'
        }
    )

    fig.update_layout(
        margin=dict(t=40, l=0, r=0, b=0),
        paper_bgcolor="white"
    )

    fig.show()

In [ ]:
# ==========================================
# CREAR GRÁFICOS
# ==========================================
generar_sunburst_deadlock(df_final)

### Conclusiones y Justificación
La razón por las que elegí estos criterios son las siguientes:
- Distribución de costos por categoría (Criterio 1): Este criterio permite visualizar la estructura de progresión de Deadlock. Al segmentar los items por Tier y Categoría (Weapon, Vitality, Spirit), podemos identificar qué ramas de especialización requieren una mayor acumulación de "almas" para alcanzar su máximo potencial en las distintas fases de la partida.

- Densidad de atributos por alma invertida (Criterio 2): Se eligió para medir la eficiencia técnica de la inversión económica. Este criterio responde a la pregunta de si los items de alto costo (Tier 3 y 4) mantienen la misma rentabilidad estadística que los items básicos, o si su valor reside principalmente en efectos únicos no cuantitativos.

Con estos datos puedo cooncluir queTras analizar la visualización generada (el Sunburst Chart), se establecen las siguientes conclusiones:

- Eficiencia decreciente en el escalado: Los items de Tier 1 presentan la densidad de atributos más alta del gráfico. Esto demuestra que el diseño del juego incentiva una fase inicial de compras rápidas para obtener picos de poder inmediatos, sacrificando eficiencia económica a medida que el jugador avanza hacia items de Tier 4.

- Especialización de la rama Spirit: Se observa que la categoría Spirit concentra una mayor proporción de su valor total en items de Tier 3 en comparación con la categoría Weapon. Esto sugiere que los héroes basados en habilidades dependen de una economía de "juego medio" (mid-game) mucho más robusta para ser competitivos.

- Balance de Vitalidad: La categoría Vitality muestra una distribución de densidad más uniforme entre Tiers, lo que indica que la supervivencia escala de forma más constante y predecible que el daño o la utilidad de habilidades a lo largo de la partida.

### Código entregado por la IA (Gemini Pro 3.1)

In [31]:
import pandas as pd
import plotly.express as px

def generar_ternario_definitivo_con_imagenes(df_crit1, df_crit2, df_benyen, df_heroes):
    """
    Genera un Gráfico Ternario unificando la composición económica de la build (Romero)
    con la eficiencia técnica de los ítems (Ferrada).
    ¡Incluye las imágenes de los héroes en el hover!
    """
    print("Calculando el balance económico del Meta y cargando imágenes...")

    # 1. Filtrar solo las builds Endgame de Romero
    df_endgame = df_crit2[df_crit2['tipo_build'] == 'Endgame'].copy()

    # 2. Calcular los porcentajes de inversión para el triángulo
    df_endgame['pct_weapon'] = (df_endgame['costo_weapon'] / df_endgame['costo_total']) * 100
    df_endgame['pct_vitality'] = (df_endgame['costo_vitality'] / df_endgame['costo_total']) * 100
    df_endgame['pct_spirit'] = (df_endgame['costo_spirit'] / df_endgame['costo_total']) * 100

    # 3. Calcular métricas combinadas (Popularidad y Densidad)
    filas_combinadas = []
    for _, row in df_endgame.iterrows():
        heroe_name = row['heroe']
        items_build = row['items_good'] # Lista de class_names

        # Obtenemos el class_name del héroe (para la imagen)
        # Buscamos en df_heroes el class_name que corresponde a este heroe_name
        match_heroe = df_heroes[df_heroes['name'] == heroe_name]
        clase_heroe = match_heroe.iloc[0]['class_name'] if not match_heroe.empty else ""

        # Datos Ferrada (Densidad)
        df_f = df_benyen[df_benyen['class_name'].isin(items_build)]
        avg_densidad = df_f['densidad_stats'].mean() if not df_f.empty else 0

        # Datos Romero (Popularidad)
        df_r = df_crit1[df_crit1['class_name'].isin(items_build)]
        avg_pop = df_r['popularidad_good'].mean() if not df_r.empty else 1

        filas_combinadas.append({
            'heroe': heroe_name,
            'clase_heroe': clase_heroe, # Guardamos el ID interno para la foto
            'avg_densidad': avg_densidad,
            'avg_popularidad': avg_pop
        })

    df_metricas = pd.DataFrame(filas_combinadas)
    df_final = pd.merge(df_endgame, df_metricas, on='heroe')

    # ---> AQUÍ AGREGAMOS LA MAGIA DE LA IMAGEN <---
    url_base = "https://assets.deadlock-api.com/images/heroes/"
    # Creamos la etiqueta HTML de la imagen usando el class_name del héroe
    df_final['img_hover'] = df_final['clase_heroe'].apply(
        lambda clase: f"<img src='{url_base}{clase}_icon.webp' width='60' height='60'>" if clase else ""
    )

    # 4. GENERACIÓN DEL GRÁFICO TERNARIO
    fig = px.scatter_ternary(
        df_final,
        a="pct_weapon",     # Eje Superior
        b="pct_vitality",   # Eje Inferior Izquierdo
        c="pct_spirit",     # Eje Inferior Derecho
        hover_name="heroe",
        size="avg_popularidad",
        color="avg_densidad",
        color_continuous_scale="Turbo",
        title="Balance del Meta: Distribución Económica y Eficiencia por Héroe",
        labels={
            "pct_weapon": "% Inversión en Weapon",
            "pct_vitality": "% Inversión en Vitality",
            "pct_spirit": "% Inversión en Spirit",
            "avg_densidad": "Eficiencia<br>(Stats/1k Almas)"
        },
        size_max=30,
        # Pasamos la imagen oculta en los datos del hover
        hover_data={"img_hover": False, "clase_heroe": False, "items_good": False}
    )

    # ---> CONFIGURAMOS EL HOVER PARA QUE MUESTRE LA IMAGEN <---
    fig.update_traces(
        # customdata[0] será nuestra columna 'img_hover'
        hovertemplate=(
            "%{customdata[0]}<br><br>" # Aquí renderiza la foto del héroe
            "<b>%{hovertext}</b><br><br>" # Nombre del héroe
            "Weapon: %{a:.1f}%<br>"
            "Vitality: %{b:.1f}%<br>"
            "Spirit: %{c:.1f}%<br>"
            "Eficiencia: %{marker.color:.1f} stats/1k almas<br>"
            "<extra></extra>" # Quita el cuadro lateral
        )
    )

    # Mejoras visuales del triángulo
    fig.update_layout(
        ternary=dict(
            sum=100,
            aaxis=dict(title="Weapon (%)", ticksuffix="%", min=0.01, linewidth=2),
            baxis=dict(title="Vitality (%)", ticksuffix="%", min=0.01, linewidth=2),
            caxis=dict(title="Spirit (%)", ticksuffix="%", min=0.01, linewidth=2),
            bgcolor="rgba(240, 240, 240, 0.8)"
        ),
        paper_bgcolor="white",
        font=dict(size=12)
    )

    fig.show()
    return df_final

# ==========================================
# CÓMO EJECUTARLO EN TU CELDA FINAL:
# ==========================================
# (Asegúrate de haber ejecutado antes df_crit1, df_crit2, df_benyen y de tener df_heroes_romero cargado)
df_mega_ternario = generar_ternario_definitivo_con_imagenes(df_crit1, df_crit2, df_benyen, df_heroes_romero)

Calculando el balance económico del Meta y cargando imágenes...
